In [0]:
# Leer las cifras actuales, sin fijar cantidades de nuestra muestra.

filas_bronze = spark.table(
    "workspace.default.bronze_pos_raw"
).count()

silver = spark.sql("""
    SELECT
        COUNT(*) AS filas,
        SUM(CASE
            WHEN store_number IS NULL
              OR fecha IS NULL
              OR check_number IS NULL
              OR begin_time IS NULL
              OR hora_inicio IS NULL
              OR error_conversion_numerica
              OR _rescued_data IS NOT NULL
            THEN 1 ELSE 0
        END) AS filas_con_problemas
    FROM workspace.default.silver_pos_lines
""").first()

pedidos = spark.sql("""
    SELECT
        COUNT(*) AS pedidos,
        COUNT(importe_comparable) AS con_importe,
        COALESCE(SUM(importe_comparable), 0) AS importe,
        SUM(CASE
            WHEN total_distintos <> 1 THEN 1 ELSE 0
        END) AS total_inconsistente
    FROM workspace.default.silver_orders
""").first()

gold = spark.sql("""
    SELECT
        COUNT(*) AS filas,
        COUNT(DISTINCT struct(store_number, fecha)) AS claves,
        COALESCE(SUM(pedidos), 0) AS pedidos,
        COALESCE(SUM(pedidos_con_importe), 0) AS con_importe,
        COALESCE(SUM(importe_estimado), 0) AS importe
    FROM workspace.default.gold_store_daily
""").first()

pedidos_revision = spark.table(
    "workspace.default.silver_orders_review"
).count()


# Comprobar que las tablas sean consistentes entre sí.

pruebas = {
    "Bronze contiene registros":
        filas_bronze > 0,

    "Silver conserva la cantidad de filas de Bronze":
        silver["filas"] == filas_bronze,

    "Silver no presenta problemas críticos":
        silver["filas_con_problemas"] == 0,

    "Cada pedido tiene un único Total informado":
        pedidos["total_inconsistente"] == 0,

    "Gold tiene una fila por tienda y fecha":
        gold["filas"] == gold["claves"],

    "Gold conserva la cantidad de pedidos":
        gold["pedidos"] == pedidos["pedidos"],

    "Gold conserva los pedidos con importe":
        gold["con_importe"] == pedidos["con_importe"],

    "Gold conserva la suma exacta del importe":
        gold["importe"] == pedidos["importe"],

    "La tabla de revisión contiene todos los pendientes":
        pedidos_revision == pedidos["pedidos"] - pedidos["con_importe"]
}

fallos = []

for nombre, resultado in pruebas.items():
    estado = "CORRECTO" if resultado else "ERROR"
    print(f"{estado}: {nombre}")

    if not resultado:
        fallos.append(nombre)

if fallos:
    raise RuntimeError(
        "Validación fallida: " + "; ".join(fallos)
    )

print(
    "Pasaron las nueve comprobaciones entre tablas. "
    "Falta comprobar los archivos en la siguiente celda."
)

In [0]:
from pyspark.sql import functions as F

auditoria = spark.table("workspace.default.audit_source_files")

ultima_revision = (
    auditoria
    .orderBy(F.col("revisado_at").desc())
    .select("ejecucion_id")
    .first()
)

if ultima_revision is None:
    raise RuntimeError("No existe un inventario de archivos.")

esperados = (
    auditoria
    .filter(
        F.col("ejecucion_id") == ultima_revision["ejecucion_id"]
    )
    .filter(F.col("estado") == "Con datos")
    .select(
        F.regexp_replace("ruta", "^dbfs:", "").alias("ruta")
    )
    .distinct()
)

cargados = (
    spark.table("workspace.default.bronze_pos_raw")
    .select(
        F.regexp_replace(
            "source_file", "^dbfs:", ""
        ).alias("ruta")
    )
    .distinct()
)

faltantes = (
    esperados
    .join(cargados, on="ruta", how="left_anti")
    .limit(10)
    .collect()
)

if faltantes:
    ejemplos = ", ".join(fila["ruta"] for fila in faltantes)
    raise RuntimeError(
        "Hay archivos con datos que no llegaron a Bronze. "
        f"Ejemplos: {ejemplos}"
    )

print("CORRECTO: todos los archivos con datos llegaron a Bronze.")
print("Validación completa: pasaron las diez comprobaciones.")